In [21]:
# Cell 1: Install required packages
%pip install python-dotenv google-generativeai python-pptx pillow requests --quiet
print("✅ Packages installed successfully!")

Note: you may need to restart the kernel to use updated packages.
✅ Packages installed successfully!


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
# First, let's check what Gemini models are available
print("🔍 Checking available Gemini models...")
try:
    for model in genai.list_models():
        if 'generateContent' in model.supported_generation_methods:
            print(f"  ✅ {model.name}")
except Exception as e:
    print(f"  ❌ Could not list models: {e}")

# Test Pexels API key
print("\n🔍 Testing Pexels API key...")
pexels_key = os.getenv('PEXELS_API_KEY')
if pexels_key:
    print(f"Pexels key found: {pexels_key[:10]}...")
    
    # Test the API
    import requests
    headers = {'Authorization': pexels_key}
    try:
        response = requests.get('https://api.pexels.com/v1/curated?per_page=1', 
                               headers=headers, timeout=10)
        print(f"Pexels API test: Status {response.status_code}")
        if response.status_code == 200:
            print("✅ Pexels API working!")
        elif response.status_code == 401:
            print("❌ Pexels API key invalid or expired")
        elif response.status_code == 429:
            print("⚠️ Pexels rate limit exceeded")
        else:
            print(f"⚠️ Pexels error: {response.status_code}")
    except Exception as e:
        print(f"❌ Pexels test error: {e}")
else:
    print("❌ No Pexels API key found in .env")

🔍 Checking available Gemini models...
  ✅ models/gemini-2.5-flash
  ✅ models/gemini-2.5-pro
  ✅ models/gemini-2.0-flash
  ✅ models/gemini-2.0-flash-001
  ✅ models/gemini-2.0-flash-exp-image-generation
  ✅ models/gemini-2.0-flash-lite-001
  ✅ models/gemini-2.0-flash-lite
  ✅ models/gemini-exp-1206
  ✅ models/gemini-2.5-flash-preview-tts
  ✅ models/gemini-2.5-pro-preview-tts
  ✅ models/gemma-3-1b-it
  ✅ models/gemma-3-4b-it
  ✅ models/gemma-3-12b-it
  ✅ models/gemma-3-27b-it
  ✅ models/gemma-3n-e4b-it
  ✅ models/gemma-3n-e2b-it
  ✅ models/gemini-flash-latest
  ✅ models/gemini-flash-lite-latest
  ✅ models/gemini-pro-latest
  ✅ models/gemini-2.5-flash-lite
  ✅ models/gemini-2.5-flash-image
  ✅ models/gemini-2.5-flash-preview-09-2025
  ✅ models/gemini-2.5-flash-lite-preview-09-2025
  ✅ models/gemini-3-pro-preview
  ✅ models/gemini-3-flash-preview
  ✅ models/gemini-3-pro-image-preview
  ✅ models/nano-banana-pro-preview
  ✅ models/gemini-robotics-er-1.5-preview
  ✅ models/gemini-2.5-computer-

In [22]:
# Cell 2: Import libraries and set up
import os
import json
import requests
from PIL import Image
from io import BytesIO
import google.generativeai as genai
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from dotenv import load_dotenv
import warnings
import hashlib
warnings.filterwarnings('ignore')

# Load environment variables from .env file
load_dotenv()

# Display current directory to verify .env location
print("Current directory:", os.getcwd())
print(".env file exists:", os.path.exists('.env'))

c:\Users\sharm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sharm\AppData\Local\Temp\ipykernel_12764\3771724743.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


c:\Users\sharm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sharm\AppData\Local\Temp\ipykernel_12764\3771724743.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Current directory: c:\satvik\legal\gen-AI-projects\ppy-generator
.env file exists: True


In [36]:
# Cell 3: WORKING PPTGenerator Class
class PPTGenerator:
    def __init__(self, gemini_api_key=None, pexels_api_key=None):
        """
        Initialize PPT Generator
        """
        # Get API keys
        self.gemini_api_key = gemini_api_key or os.getenv('GEMINI_API_KEY')
        self.pexels_api_key = pexels_api_key or os.getenv('PEXELS_API_KEY')
        
        if not self.gemini_api_key:
            raise ValueError("❌ Gemini API key is required!")
        
        # Configure Gemini
        genai.configure(api_key=self.gemini_api_key)
        
        print("🔧 Initializing Gemini AI...")
        
        # Use a model that's confirmed to work
        try:
            # Try gemini-2.0-flash first (fast and reliable)
            self.model = genai.GenerativeModel('models/gemini-2.0-flash')
            print("✅ Using: models/gemini-2.0-flash")
        except:
            try:
                # Fallback to gemini-pro-latest
                self.model = genai.GenerativeModel('models/gemini-pro-latest')
                print("✅ Using: models/gemini-pro-latest")
            except Exception as e:
                print(f"❌ Model initialization failed: {e}")
                raise
        
        # Initialize presentation
        self.presentation = Presentation()
        self.presentation.slide_width = Inches(13.33)
        self.presentation.slide_height = Inches(7.5)
        
        # Create temp directory
        self.temp_dir = "ppt_images"
        os.makedirs(self.temp_dir, exist_ok=True)
        
        print("\n" + "="*50)
        print("🎉 PPT Generator Initialized!")
        print(f"   Gemini: ✓")
        print(f"   Pexels: {'✓' if self.pexels_api_key else '✗'}")
        print("="*50 + "\n")
    
    def generate_slide_content(self, topic):
        """Generate slide content - SIMPLE AND RELIABLE"""
        prompt = f"""
        Create a presentation about "{topic}" with exactly 6 slides.
        
        Return ONLY this JSON format:
        {{
          "title": "Presentation Title",
          "subtitle": "Brief subtitle",
          "slides": [
            {{
              "index": 1,
              "title": "Cover: {topic}",
              "type": "cover",
              "key_bullets": ["Introduction to topic"],
              "image_prompt": "technology"
            }},
            {{
              "index": 2,
              "title": "Agenda",
              "type": "agenda",
              "key_bullets": ["What we'll cover"],
              "image_prompt": "agenda"
            }},
            {{
              "index": 3,
              "title": "Key Concepts",
              "type": "content",
              "key_bullets": ["Concept 1", "Concept 2"],
              "image_prompt": "concepts"
            }},
            {{
              "index": 4,
              "title": "Applications",
              "type": "content", 
              "key_bullets": ["Use Case 1", "Use Case 2"],
              "image_prompt": "applications"
            }},
            {{
              "index": 5,
              "title": "Benefits",
              "type": "content",
              "key_bullets": ["Benefit 1", "Benefit 2"],
              "image_prompt": "benefits"
            }},
            {{
              "index": 6,
              "title": "Conclusion",
              "type": "conclusion",
              "key_bullets": ["Summary", "Next Steps"],
              "image_prompt": "conclusion"
            }}
          ]
        }}
        
        Keep image_prompt to 1-2 simple words.
        """
        
        try:
            print(f"🤖 Generating content for: {topic}")
            
            response = self.model.generate_content(prompt)
            content = response.text.strip()
            
            print(f"📝 Response received ({len(content)} chars)")
            
            # Clean JSON
            import re
            # Find JSON pattern
            json_pattern = r'\{.*\}'
            match = re.search(json_pattern, content, re.DOTALL)
            
            if match:
                json_str = match.group(0)
                try:
                    data = json.loads(json_str)
                    print(f"✅ Generated {len(data.get('slides', []))} slides")
                    return data
                except json.JSONDecodeError as e:
                    print(f"❌ JSON parse error: {e}")
            else:
                print("❌ No JSON found in response")
            
            # Fallback
            return self._create_fallback_content(topic)
            
        except Exception as e:
            print(f"❌ Generation error: {e}")
            return self._create_fallback_content(topic)
    
    def _create_fallback_content(self, topic):
        """Create fallback content"""
        print("🔄 Using fallback content...")
        return {
            "title": topic,
            "subtitle": "A Professional Presentation",
            "slides": [
                {"index": 1, "title": topic, "type": "cover", "key_bullets": ["Welcome", "Overview"], "image_prompt": "technology"},
                {"index": 2, "title": "Agenda", "type": "agenda", "key_bullets": ["Introduction", "Key Points", "Conclusion"], "image_prompt": "planning"},
                {"index": 3, "title": "Introduction", "type": "content", "key_bullets": ["Background", "Importance", "Scope"], "image_prompt": "introduction"},
                {"index": 4, "title": "Main Concepts", "type": "content", "key_bullets": ["Concept 1", "Concept 2", "Concept 3"], "image_prompt": "ideas"},
                {"index": 5, "title": "Implementation", "type": "content", "key_bullets": ["Steps", "Requirements", "Timeline"], "image_prompt": "implementation"},
                {"index": 6, "title": "Conclusion", "type": "conclusion", "key_bullets": ["Summary", "Key Takeaways", "Q&A"], "image_prompt": "success"}
            ]
        }
    
    def download_from_pexels(self, query, filename=None):
        """Download image from Pexels - IMPROVED"""
        if not self.pexels_api_key:
            print(f"⚠️ No Pexels API key")
            return self._create_placeholder_image(query)
        
        try:
            # Simplify query
            simple_query = query.split()[0] if query.split() else query
            simple_query = simple_query[:20].lower()
            
            # Create filename
            if not filename:
                import hashlib
                safe_name = "".join(c if c.isalnum() else "_" for c in simple_query[:20])
                hash_id = hashlib.md5(simple_query.encode()).hexdigest()[:6]
                filename = f"{safe_name}_{hash_id}.jpg"
            
            filepath = os.path.join(self.temp_dir, filename)
            
            # Check cache
            if os.path.exists(filepath):
                print(f"♻️ Cached: {filename}")
                return filepath
            
            # Pexels API
            headers = {'Authorization': self.pexels_api_key}
            params = {
                'query': simple_query,
                'per_page': 5,  # Try more results
                'orientation': 'landscape',
                'size': 'medium'
            }
            
            print(f"🔍 Searching Pexels: '{simple_query}'")
            
            response = requests.get(
                'https://api.pexels.com/v1/search',
                headers=headers,
                params=params,
                timeout=15
            )
            
            if response.status_code != 200:
                print(f"⚠️ Pexels API error {response.status_code}")
                return self._create_placeholder_image(query)
            
            data = response.json()
            
            if not data.get('photos'):
                print(f"❌ No photos for '{simple_query}'")
                return self._create_placeholder_image(query)
            
            # Try each photo
            for i, photo in enumerate(data['photos']):
                try:
                    image_url = photo.get('src', {}).get('medium') or photo.get('src', {}).get('large')
                    
                    if not image_url:
                        continue
                    
                    print(f"⬇️ Trying image {i+1}...")
                    img_response = requests.get(image_url, timeout=15)
                    img_response.raise_for_status()
                    
                    # Save
                    with open(filepath, 'wb') as f:
                        f.write(img_response.content)
                    
                    # Verify
                    try:
                        with Image.open(filepath) as img:
                            img.verify()
                        print(f"✅ Downloaded: {filename}")
                        print(f"   📷 Photo by: {photo.get('photographer', 'Unknown')}")
                        return filepath
                    except:
                        continue
                        
                except Exception as e:
                    print(f"  ⚠️ Download failed: {e}")
                    continue
            
            print(f"❌ All downloads failed for '{simple_query}'")
            return self._create_placeholder_image(query)
            
        except Exception as e:
            print(f"❌ Pexels error: {e}")
            return self._create_placeholder_image(query)
    
    def _create_placeholder_image(self, query):
        """Create placeholder image"""
        try:
            from PIL import Image, ImageDraw, ImageFont
            
            # Simple filename
            safe_name = "".join(c if c.isalnum() else "_" for c in query[:20])
            filename = f"placeholder_{safe_name}.jpg"
            filepath = os.path.join(self.temp_dir, filename)
            
            # Create image
            img = Image.new('RGB', (800, 600), color='#4A90E2')
            draw = ImageDraw.Draw(img)
            
            # Try to add text
            try:
                # Try default font
                font = ImageFont.load_default()
                text = f"{query[:30]}"
                # Center text
                from PIL import ImageFont
                text_width = draw.textlength(text, font=font)
                position = ((800 - text_width) // 2, 250)
                draw.text(position, text, fill='white', font=font)
                
                # Add "Image Placeholder"
                draw.text((50, 50), "Image Placeholder", fill='white')
            except:
                pass
            
            img.save(filepath)
            print(f"📄 Created placeholder: {filename}")
            return filepath
            
        except Exception as e:
            print(f"❌ Placeholder failed: {e}")
            return None
    
    def create_slide(self, slide_data):
        """Create a single slide"""
        try:
            slide_type = slide_data.get('type', 'content')
            title = slide_data.get('title', 'Slide')
            bullets = slide_data.get('key_bullets', [])
            
            # Choose layout
            if slide_type == 'cover':
                layout_idx = 0  # Title slide
            else:
                layout_idx = 1  # Title and content
            
            # Create slide
            slide_layout = self.presentation.slide_layouts[layout_idx]
            slide = self.presentation.slides.add_slide(slide_layout)
            
            # Set title
            if slide.shapes.title:
                title_shape = slide.shapes.title
                title_shape.text = title
                
                # Style title
                title_frame = title_shape.text_frame
                title_paragraph = title_frame.paragraphs[0]
                title_paragraph.font.size = Pt(36 if slide_type == 'cover' else 28)
                title_paragraph.font.bold = True
            
            # Add content for non-cover slides
            if slide_type != 'cover' and len(slide.placeholders) > 1:
                content_placeholder = slide.placeholders[1]
                text_frame = content_placeholder.text_frame
                text_frame.clear()
                
                for bullet in bullets:
                    p = text_frame.add_paragraph()
                    p.text = f"• {bullet}"
                    p.level = 0
                    p.font.size = Pt(20)
                    p.space_before = Pt(8)
            
            # Add image for content slides
            if slide_type not in ['cover', 'agenda']:
                image_prompt = slide_data.get('image_prompt', title)
                image_path = self.download_from_pexels(image_prompt)
                
                if image_path and os.path.exists(image_path):
                    try:
                        # Add to right side
                        left = Inches(7)
                        top = Inches(1.5)
                        width = Inches(5)
                        slide.shapes.add_picture(image_path, left, top, width=width)
                        print(f"   🖼️ Added: {os.path.basename(image_path)}")
                    except Exception as e:
                        print(f"   ⚠️ Image error: {e}")
            
            return slide
            
        except Exception as e:
            print(f"❌ Slide creation error: {e}")
            # Create simple slide as fallback
            try:
                slide = self.presentation.slides.add_slide(self.presentation.slide_layouts[1])
                if slide.shapes.title:
                    slide.shapes.title.text = title
                return slide
            except:
                return None
    
    def generate_presentation(self, topic):
        """Generate complete presentation"""
        print("\n" + "="*50)
        print(f"🚀 GENERATING: {topic}")
        print("="*50)
        
        # Get content
        content = self.generate_slide_content(topic)
        
        # Create slides
        print("\n📊 Creating slides...")
        successful = 0
        
        for slide_data in content.get('slides', []):
            try:
                idx = slide_data.get('index', 0)
                title = slide_data.get('title', 'Slide')[40:]
                print(f"  Slide {idx}: {title}...")
                
                slide = self.create_slide(slide_data)
                if slide:
                    successful += 1
            except Exception as e:
                print(f"  ❌ Failed: {e}")
        
        print(f"\n✅ Created {successful}/{len(content.get('slides', []))} slides")
        return self.presentation
    
    def save(self, filename=None):
        """Save presentation"""
        if not filename:
            import datetime
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
            filename = f"presentation_{timestamp}.pptx"
        
        try:
            self.presentation.save(filename)
            filepath = os.path.abspath(filename)
            size_kb = os.path.getsize(filename) / 1024
            
            print(f"\n💾 SAVED: {filename}")
            print(f"📁 Path: {filepath}")
            print(f"📊 Size: {size_kb:.1f} KB")
            print(f"📈 Slides: {len(self.presentation.slides)}")
            
            return filename
            
        except Exception as e:
            print(f"❌ Save error: {e}")
            return None
    
    def preview(self):
        """Preview information"""
        print(f"\n📊 Presentation Info:")
        print(f"   Slides: {len(self.presentation.slides)}")
        
        # Show downloaded images
        import glob
        images = glob.glob(os.path.join(self.temp_dir, "*.jpg"))
        if images:
            print(f"   Images: {len(images)} downloaded")
            for img in images[:3]:
                print(f"     • {os.path.basename(img)}")

In [45]:
# Cell 4: Test if .env is working
print("🔍 Checking .env file...")
print(f"GEMINI_API_KEY loaded: {'Yes' if os.getenv('GEMINI_API_KEY') else 'No'}")
print(f"PEXELS_API_KEY loaded: {'Yes' if os.getenv('PEXELS_API_KEY') else 'No'}")

# If keys aren't loaded, show help
if not os.getenv('GEMINI_API_KEY'):
    print("\n⚠️  GEMINI_API_KEY not found in .env file!")
    print("Make sure your .env file contains:")
    print("GEMINI_API_KEY=your_actual_key_here")
    print("PEXELS_API_KEY=your_actual_key_here")
    print("\nPlace .env file in:", os.getcwd())

🔍 Checking .env file...
GEMINI_API_KEY loaded: Yes
PEXELS_API_KEY loaded: Yes


In [46]:
# Initialize
generator = PPTGenerator()

# Generate
topic = "Artificial Intelligence in Healthcare"
presentation = generator.generate_presentation(topic)

# Save
generator.save("AI_Healthcare_Presentation.pptx")



🔧 Initializing Gemini AI...
✅ Using: models/gemini-2.0-flash

🎉 PPT Generator Initialized!
   Gemini: ✓
   Pexels: ✓


🚀 GENERATING: Artificial Intelligence in Healthcare
🤖 Generating content for: Artificial Intelligence in Healthcare
❌ Generation error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 38.717873767s. [links {
  description: "Learn more about Gemini API quota

'AI_Healthcare_Presentation.pptx'

In [34]:
# Cell 7: Save the presentation
saved_file = generator.save()
print(f"\n📂 File saved at: {os.path.abspath(saved_file)}")


💾 Presentation saved successfully!
📂 File: presentation_20260128_145241.pptx
📁 Path: c:\satvik\legal\gen-AI-projects\ppy-generator\presentation_20260128_145241.pptx
📊 Size: 34.0 KB
📈 Slides: 8

📂 File saved at: c:\satvik\legal\gen-AI-projects\ppy-generator\presentation_20260128_145241.pptx


In [ ]:
# Cell 10: Batch generate multiple presentations
def batch_generate(topics):
    """Generate multiple presentations"""
    results = []
    for topic in topics:
        print(f"\n{'='*50}")
        print(f"Processing: {topic}")
        print('='*50)
        
        # Create new generator for each topic
        generator = PPTGenerator()
        
        # Generate and save
        generator.generate_presentation(topic)
        filename = generator.save(f"{topic[:20]}.pptx")
        
        results.append({
            'topic': topic,
            'filename': filename,
            'slides': len(generator.presentation.slides)
        })
    
    print(f"\n✅ Batch generation complete!")
    print(f"   Generated {len(results)} presentations")
    
    return results

# Example batch topics
sample_topics = [
    "Digital Marketing Trends 2024",
    "Cybersecurity Best Practices",
    "Remote Work Productivity"
]

# Uncomment to run batch
# batch_results = batch_generate(sample_topics)